# PyTorch 性能计时与 Profiler

## 学习目标

正确测量前向与训练步骤耗时，理解预热、设备同步、推理模式、batch size 和 profiler 的基本使用方法。

## 概念模型

性能测量必须固定输入、重复运行并区分 CPU 提交时间与设备实际执行时间。先用简单计时确认问题，再用 profiler 定位算子。

In [ ]:
import time
import torch
from torch import nn

torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = nn.Sequential(nn.Linear(128, 256), nn.ReLU(), nn.Linear(256, 64)).to(device)
inputs = torch.randn(64, 128, device=device)
print('device:', device)

### 实验 1：预热、同步与重复计时

**实验目的**：通过预热排除初始化开销，重复前向降低单次噪声，并在 CUDA 计时边界同步。最终时间应除以迭代次数，报告单次延迟或吞吐。

CPU 与 CUDA 计时口径不同；性能测试还应固定线程、batch、dtype 和输入 shape。


In [ ]:
def synchronize():
    if device.type == 'cuda':
        torch.cuda.synchronize()

with torch.inference_mode():
    for _ in range(5):
        model(inputs)
    synchronize()
    started = time.perf_counter()
    for _ in range(30):
        model(inputs)
    synchronize()
elapsed = time.perf_counter() - started
print('average forward ms:', round(elapsed / 30 * 1000, 3))
assert elapsed > 0

### 实验 2：训练模式与推理模式

**实验目的**：比较普通训练前向与 eval+inference_mode。当前模型没有 Dropout/BatchNorm，所以数值应一致，但推理输出不连接计算图、开销更低。

若模型含模式相关层，数值不一定相同；性能比较必须明确是训练、纯前向还是端到端推理。


In [ ]:
model.train()
training_output = model(inputs)
assert training_output.requires_grad
model.eval()
with torch.inference_mode():
    inference_output = model(inputs)
assert not inference_output.requires_grad
torch.testing.assert_close(training_output.detach(), inference_output)
print('autograd flags:', training_output.requires_grad, inference_output.requires_grad)

### 实验 3：最小 profiler

**实验目的**：记录 CPU/CUDA 活动和输入 shape，查看最耗时算子。Profiler 自身有开销，适合定位瓶颈，不应用其结果直接替代低开销基准。

关注 self time、总时间、调用次数和 shape；GPU 分析还需预热并理解异步 kernel。


In [ ]:
activities = [torch.profiler.ProfilerActivity.CPU]
if device.type == 'cuda':
    activities.append(torch.profiler.ProfilerActivity.CUDA)
with torch.profiler.profile(activities=activities, record_shapes=True) as profile:
    with torch.inference_mode():
        for _ in range(3):
            model(inputs)
table = profile.key_averages().table(sort_by='self_cpu_time_total', row_limit=5)
print(table)
assert 'Self CPU' in table

## 检查点

解释为什么 CUDA 计时需要同步、为什么需要预热，以及 `eval()` 和 `inference_mode()` 的区别。说明 profiler 和端到端 benchmark 分别回答什么问题。

## 试一试

比较 batch size 1、16、64 的单样本平均耗时；在 CUDA 环境记录 AMP 开启前后的时间和峰值显存。

## 常见错误与调试

只运行一次就下结论、CUDA 计时未同步、把数据加载时间与模型前向混在一起、比较不同输入 shape、认为 profiler 本身没有额外开销。